In [19]:
import torch

In [20]:
x = torch.arange(
    4.0,
    requires_grad=True,
)

print("x:", x)
print("x shape:", x.shape)
print("requires_grad:", x.requires_grad)
print("Gradient before backward:", x.grad)

x: tensor([0., 1., 2., 3.], requires_grad=True)
x shape: torch.Size([4])
requires_grad: True
Gradient before backward: None


In [21]:
# y = 2xᵀx를 계산
# torch.dot(x, x)의 출력이 scalar이므로 y의 shape도 ()

y = 2 * torch.dot(x, x)

print("y:", y)
print("y shape:", y.shape)
print("Gradient function:", y.grad_fn)

y: tensor(28., grad_fn=<MulBackward0>)
y shape: torch.Size([])
Gradient function: <MulBackward0 object at 0x782eb21474f0>


In [22]:
# y에서 x까지 계산 그래프를 역방향으로 이동하며 gradient를 계산
# y가 scalar 이므로 backward()의 시작 gradient는 자동으로 1이 됨.
y.backward()

expected_gradient = 4 * x.detach()

print("x.grad:", x.grad)
print("Expected gradient:", expected_gradient)


x.grad: tensor([ 0.,  4.,  8., 12.])
Expected gradient: tensor([ 0.,  4.,  8., 12.])


In [23]:
# PyTorch는 gradient를 자동 초기화하지 않으므로 직접 0으로 만든다.
x.grad.zero_()

# 새로운 함수 y=sum(x)를 계산하고 새 계산 그래프를 만든다.
y = x.sum()
y.backward()

print("Gradient of sum(x):", x.grad)

assert torch.equal(
    x.grad,
    torch.ones_like(x),
)

Gradient of sum(x): tensor([1., 1., 1., 1.])


In [24]:
# 이번에는 gradient를 초기화하지 않고 다른 backward를 실행
# 기존 gradient 1에 d(sum(x²))/dx=2x가 더해진다. [0, 2, 4, 6]

additional_output = (x**2).sum()
additional_output.backward()

expected_accumulated_gradient = (
    torch.ones_like(x)
    + 2 * x.detach()
)

print("Accumulated gradient:", x.grad)
print("Expected gradient:", expected_accumulated_gradient)

assert torch.equal(
    x.grad,
    expected_accumulated_gradient,
)


Accumulated gradient: tensor([1., 3., 5., 7.])
Expected gradient: tensor([1., 3., 5., 7.])
